# Evaluation Matrix (3 Environments)
Notebook này gồm 3 môi trường phân lập để thử nghiệm hệ thống sử dụng **Full Groq API**.
**Lưu ý:** Hiện tại các cell đang cấu hình chạy thử nghiệm với `[:3]` (3 test cases đầu tiên). Khi mọi thứ đã ổn, **bạn chỉ cần xóa chữ `[:3]`** ở trong vòng lặp `for` của mỗi cell để quét toàn bộ 50 ca!

In [10]:
# Cell 1: Khởi tạo, khai báo Data và thư viện
import sys
import os
import json
import asyncio
import io
import contextlib
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
backend_dir = Path(os.getcwd()).parent.parent
if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

from ai_engine.hospital_graph import hospital_app as fsm_app
from ai_engine.agents.llm_service import generate_text_async
import ai_engine.agents.orchestrator_node

DATA_DIR = backend_dir / "benchmarks" / "data"
PATIENTS_PATH = DATA_DIR / "synthetic_patients.json"

with open(PATIENTS_PATH, "r", encoding="utf-8") as f:
    TEST_CASES = json.load(f)
    
print(f"Đã tải {len(TEST_CASES)} hồ sơ bệnh án chuẩn bị cho kiểm thử.")

async def simulate_user(persona, initial_msg, history, bot_msg):
    prompt = f"Bạn là một người dùng đang sử dụng Chatbot Tâm lý.\nTính cách: {persona}\nHoàn cảnh ban đầu: {initial_msg}\n\nNhiệm vụ: Trả lời lại bot trong 1-2 câu ngắn gọn, tự nhiên, chân thật theo hoàn cảnh.\nLịch sử hiện tại: {history}\nTin nhắn từ Bot: '{bot_msg}'\nBạn:"
    return await generate_text_async(model="slm", contents=prompt, model_type="slm")

def generate_dass21_array(dass21_scores: dict) -> str:
    ans = [0]*21
    def spread(total, count):
        res = [0]*count
        for i in range(total):
            res[i%count] += 1
            if res[i%count] > 3: res[i%count] = 3
        return res
        
    s_sum = max(0, dass21_scores.get("stress", 0) // 2)
    a_sum = max(0, dass21_scores.get("anxiety", 0) // 2)
    d_sum = max(0, dass21_scores.get("depression", 0) // 2)
    
    s_vals = spread(s_sum, 7)
    a_vals = spread(a_sum, 7)
    d_vals = spread(d_sum, 7)
    
    for i, idx in enumerate([0, 5, 7, 10, 11, 13, 17]): ans[idx] = s_vals[i]
    for i, idx in enumerate([1, 3, 6, 8, 14, 18, 19]): ans[idx] = a_vals[i]
    for i, idx in enumerate([2, 4, 9, 12, 15, 16, 20]): ans[idx] = d_vals[i]
        
    return "[DASS21_SUBMIT]: " + ",".join(map(str, ans))


# [TÍCH HỢP] Biến toàn cục dùng chung bộ nhớ để trữ hội thoại thuần
GLOBAL_TRANSCRIPTS = {}


Đã tải 50 hồ sơ bệnh án chuẩn bị cho kiểm thử.


In [ ]:
# Cell 1.5: Pre-generate User Turns (dùng chung cho cả 3 môi trường)
SHARED_USER_TURNS = {}

for tc in TEST_CASES[:3]:  # >>> ĐỂ CHẠY FULL, HÃY XÓA CHỮ [:3]
    uid = tc["id"]
    persona = tc.get("persona_prompt", "")
    msg = tc.get("initial_message", tc.get("core_issue", "Tôi mệt mỏi"))
    hist = ""
    turns = []
    for turn in range(3):
        user_reply = await simulate_user(persona, msg, hist, "[neutral]")
        user_reply = user_reply.strip()
        turns.append(user_reply)
        hist += f"User: {user_reply}\n"
        msg = user_reply
    SHARED_USER_TURNS[uid] = turns
    print(f"[Pre-gen] {uid}: {len(turns)} turns sẵn sàng")

print(f"✅ Đã pre-generate user turns cho {len(SHARED_USER_TURNS)} ca bệnh.")


[Pre-gen] idx01_CRISIS_7a02: 3 turns sẵn sàng
[Pre-gen] idx02_CRISIS_80ef: 3 turns sẵn sàng
[Pre-gen] idx03_CRISIS_3273: 3 turns sẵn sàng
[Pre-gen] idx04_CRISIS_b50d: 3 turns sẵn sàng
[Pre-gen] idx05_CRISIS_e93d: 3 turns sẵn sàng
[Pre-gen] idx06_BA_68f3: 3 turns sẵn sàng
[Pre-gen] idx07_BA_0b15: 3 turns sẵn sàng
[Pre-gen] idx08_BA_4b2c: 3 turns sẵn sàng
[Pre-gen] idx09_BA_7bc4: 3 turns sẵn sàng
[Pre-gen] idx10_BA_84c2: 3 turns sẵn sàng
[Pre-gen] idx11_BA_797e: 3 turns sẵn sàng
[Pre-gen] idx12_BA_5918: 3 turns sẵn sàng
[Pre-gen] idx13_BA_a7da: 3 turns sẵn sàng
[Pre-gen] idx14_BA_0d8d: 3 turns sẵn sàng
[Pre-gen] idx15_BA_3533: 3 turns sẵn sàng
[Pre-gen] idx16_MBI_8365: 3 turns sẵn sàng
[Pre-gen] idx17_MBI_19b6: 3 turns sẵn sàng
[Pre-gen] idx18_MBI_578c: 3 turns sẵn sàng
[Pre-gen] idx19_MBI_f60a: 3 turns sẵn sàng
[Pre-gen] idx20_MBI_2519: 3 turns sẵn sàng
✅ Đã pre-generate user turns cho 20 ca bệnh.


In [12]:
# Cell 2: Môi trường 1 - BASE MODEL (Zero-shot LLM)
BASE_LOGS_PATH = DATA_DIR / "base_model_logs.txt"

async def run_base_model():
    print("\n🚀 Đang chạy 1. Base Model...")
    all_logs = []
    
    for i, tc in enumerate(TEST_CASES[:20]): # >>> ĐỂ CHẠY FULL, HÃY XÓA CHỮ [:3]
        uid = tc["id"]
        msg = tc.get("initial_message", tc.get("core_issue", "Tôi mệt mỏi"))
        
        hist = ""
        log_content = f"============= BASE MODEL: CA {uid} =============\nUSER: {msg}\n"
        
        if uid not in GLOBAL_TRANSCRIPTS: GLOBAL_TRANSCRIPTS[uid] = {}
        GLOBAL_TRANSCRIPTS[uid]["question"] = msg
        pure_trans = ""
        
        for turn in range(3):
            prompt = f"Lịch sử:\n{hist}\nUser: {msg}\nAssistant:"
            bot_reply = await generate_text_async(model="slm", contents=prompt, model_type="slm")
            bot_reply = bot_reply.strip()
            
            log_content += f"BASE_LLM: {bot_reply}\n"
            pure_trans += f"{bot_reply}\n\n"
            hist += f"User: {msg}\nAssistant: {bot_reply}\n"
            
            msg = SHARED_USER_TURNS[uid][turn]
            log_content += f"USER: {msg}\n"

        all_logs.append(log_content + "\n\n")
        GLOBAL_TRANSCRIPTS[uid]["Base_LLM"] = pure_trans
        print(f"[Xong Base] Case {uid}")
        
    with open(BASE_LOGS_PATH, "w", encoding="utf-8") as f:
        f.write("".join(all_logs))
    print(f"✅ Hoàn tất Base Model. Kết quả thô lưu tại: {BASE_LOGS_PATH}")

await run_base_model()



🚀 Đang chạy 1. Base Model...
[Xong Base] Case idx01_CRISIS_7a02
[Xong Base] Case idx02_CRISIS_80ef
[Xong Base] Case idx03_CRISIS_3273
[Xong Base] Case idx04_CRISIS_b50d
[Xong Base] Case idx05_CRISIS_e93d
[Xong Base] Case idx06_BA_68f3
[Xong Base] Case idx07_BA_0b15
[Xong Base] Case idx08_BA_4b2c
[Xong Base] Case idx09_BA_7bc4
[Xong Base] Case idx10_BA_84c2
[Xong Base] Case idx11_BA_797e
[Xong Base] Case idx12_BA_5918
[Xong Base] Case idx13_BA_a7da
[Xong Base] Case idx14_BA_0d8d
[Xong Base] Case idx15_BA_3533
[Xong Base] Case idx16_MBI_8365
[Xong Base] Case idx17_MBI_19b6
[Xong Base] Case idx18_MBI_578c
[Xong Base] Case idx19_MBI_f60a
[Xong Base] Case idx20_MBI_2519
✅ Hoàn tất Base Model. Kết quả thô lưu tại: /home/xoai/Pysychology chatbot/cbt-ai-therapist-project/backend/benchmarks/data/base_model_logs.txt


In [13]:
# Cell 3: Môi trường 2 - 1-1 SYSTEM (Hệ thống ngắm bắn chặn Peer)
ONE_ON_ONE_LOGS_PATH = DATA_DIR / "1_1_system_logs.txt"

async def chat_agent_prompt_therapist(history, user_msg):
    prompt = f"""Bạn là một Bác sĩ tâm lý trị liệu chung (Biết cả CBT, MBI, BA).
Hãy lắng nghe, thấu cảm và đưa định hướng thích hợp. Luôn thông báo mình là AI, bảo vệ bí mật thông tin của khách hàng. Không khuyên những điều gây hại.
Lịch sử:\n{history}
User: {user_msg}
Bác sĩ:"""
    return await generate_text_async(model="slm", contents=prompt, model_type="slm")

async def run_1_1_system():
    print("\n🚀 Đang chạy 2. 1-1 System...")
    all_logs = []
    
    for i, tc in enumerate(TEST_CASES[:20]): # >>> ĐỂ CHẠY FULL, HÃY XÓA CHỮ [:3]
        uid = tc["id"]
        msg = tc.get("initial_message", tc.get("core_issue", "Tôi mệt mỏi"))
        
        print(f"[Xong 1-1] Case {uid}")
        log_content = f"============= 1-1 SYSTEM: CA {uid} =============\n"
        
        if uid not in GLOBAL_TRANSCRIPTS: GLOBAL_TRANSCRIPTS[uid] = {}
        GLOBAL_TRANSCRIPTS[uid]["question"] = msg
        pure_trans = ""
        hist = ""
        
        for turn in range(3):
            bot_reply = await chat_agent_prompt_therapist(hist, msg)
            bot_reply = bot_reply.strip()
            
            log_content += f"USER: {msg}\nSYSTEM_1_1: {bot_reply}\n"
            pure_trans += f"{bot_reply}\n\n"
            hist += f"User: {msg}\nBác sĩ: {bot_reply}\n"
            
            msg = SHARED_USER_TURNS[uid][turn]

        all_logs.append(log_content + "\n\n")
        GLOBAL_TRANSCRIPTS[uid]["Prompt_Therapist"] = pure_trans

    with open(ONE_ON_ONE_LOGS_PATH, "w", encoding="utf-8") as f:
        f.write("".join(all_logs))
    print(f"✅ Hoàn tất 1-1 System. Kết quả thô lưu tại: {ONE_ON_ONE_LOGS_PATH}")

await run_1_1_system()



🚀 Đang chạy 2. 1-1 System...
[Xong 1-1] Case idx01_CRISIS_7a02
[Xong 1-1] Case idx02_CRISIS_80ef
[Xong 1-1] Case idx03_CRISIS_3273
[Xong 1-1] Case idx04_CRISIS_b50d
[Xong 1-1] Case idx05_CRISIS_e93d
[Xong 1-1] Case idx06_BA_68f3
[Xong 1-1] Case idx07_BA_0b15
[Xong 1-1] Case idx08_BA_4b2c
[Xong 1-1] Case idx09_BA_7bc4
[Xong 1-1] Case idx10_BA_84c2
[Xong 1-1] Case idx11_BA_797e
[Xong 1-1] Case idx12_BA_5918
[Xong 1-1] Case idx13_BA_a7da
[Xong 1-1] Case idx14_BA_0d8d
[Xong 1-1] Case idx15_BA_3533
[Xong 1-1] Case idx16_MBI_8365
[Xong 1-1] Case idx17_MBI_19b6
[Xong 1-1] Case idx18_MBI_578c
[Xong 1-1] Case idx19_MBI_f60a
[Xong 1-1] Case idx20_MBI_2519
✅ Hoàn tất 1-1 System. Kết quả thô lưu tại: /home/xoai/Pysychology chatbot/cbt-ai-therapist-project/backend/benchmarks/data/1_1_system_logs.txt


In [14]:
# Cell 4: Môi trường 3 - MULTI-AGENT SYSTEM (Nguyên mẫu hệ thống đầy đủ)
MULTI_AGENT_LOGS_PATH = DATA_DIR / "multi_agent_system_logs.txt"

async def run_multi_agent():
    print("\n🚀 Đang chạy 3. Multi-Agent System...")
    all_logs = []
    
    for i, tc in enumerate(TEST_CASES[:20]): # >>> ĐỂ CHẠY FULL, HÃY XÓA CHỮ [:3]
        uid = tc["id"]
        msg = tc.get("initial_message", tc.get("core_issue", "Tôi mệt mỏi"))
        
        print(f"[Xong Multi-Agent] Case {uid}")
        f = io.StringIO()
        log_content = f"============= MULTI-AGENT SYSTEM: CA {uid} =============\n"
        
        if uid not in GLOBAL_TRANSCRIPTS: GLOBAL_TRANSCRIPTS[uid] = {}
        GLOBAL_TRANSCRIPTS[uid]["question"] = msg
        pure_trans = ""
        current_fsm_route = tc.get("therapy_route", "CBT")
        shared_turn = 0
        
        initial_state = {
            "user_message": msg,
            "user_name": tc.get("name", "User"),
            "selected_model": "slm",
            "active_protocol": None  # Skip ClinicalAssessor trong benchmark
        }
        config = {"configurable": {"thread_id": f"sim_multi_{uid}"}}
        
        with contextlib.redirect_stdout(f):
            for turn in range(3):
                state_update = initial_state if turn == 0 else {"user_message": msg}
                res = await fsm_app.ainvoke(state_update, config=config)
                bot_reply = res.get("final_reply", "")
                
                log_content += f"USER: {msg}\nOURS_MULTI: {bot_reply}\n"
                pure_trans += f"{bot_reply}\n\n"
                ui_action = res.get("ui_action", "NONE")
                if ui_action == "SHOW_DASS21":
                    msg = generate_dass21_array(tc.get("dass21", {}))
                else:
                    msg = SHARED_USER_TURNS[uid][shared_turn]
                    shared_turn += 1
                msg = msg.strip()

        all_logs.append(log_content + "\n[TERMINAL RUNTIME LOGS NGẦM (NHẬT KÝ ĐẦY ĐỦ)]:\n" + f.getvalue() + "\n\n")
        GLOBAL_TRANSCRIPTS[uid]["Ours_FSM_Multi"] = pure_trans
        GLOBAL_TRANSCRIPTS[uid]["fsm_route"] = current_fsm_route

    with open(MULTI_AGENT_LOGS_PATH, "w", encoding="utf-8") as f:
        f.write("".join(all_logs))
    print(f"✅ Hoàn tất Multi-Agent System. Kết quả thô lưu tại: {MULTI_AGENT_LOGS_PATH}")

await run_multi_agent()



🚀 Đang chạy 3. Multi-Agent System...
[Xong Multi-Agent] Case idx01_CRISIS_7a02
[Xong Multi-Agent] Case idx02_CRISIS_80ef
[Xong Multi-Agent] Case idx03_CRISIS_3273


CancelledError: 

## Tổng hợp (Aggregation & Export)
Bước cuối: Gộp kết quả hội thoại thuần từ 3 môi trường vào chung file `generated_transcripts.json` để Pipeline máy chấm tương thích chạy.

In [ ]:
# Cell 5: Kết xuất file Benchmark JSON cuối cùng
TRANSCRIPTS_PATH = DATA_DIR / "generated_transcripts.json"
generated_data = []

for uid, data in GLOBAL_TRANSCRIPTS.items():
    generated_data.append({
        "case_id": uid,
        "question": data.get("question", ""),
        "fsm_route": data.get("fsm_route", "CBT"),
        "transcripts": {
            "Base_LLM": data.get("Base_LLM", "").strip(),
            "Prompt_Therapist": data.get("Prompt_Therapist", "").strip(),
            "Ours_FSM_Multi": data.get("Ours_FSM_Multi", "").strip()
        }
    })

with open(TRANSCRIPTS_PATH, "w", encoding="utf-8") as f:
    json.dump(generated_data, f, ensure_ascii=False, indent=2)

print(f"✅ ĐÃ HOÀN TẤT TỔNG HỢP {len(generated_data)} CA BỆNH ÁN ẢO.")
print(f"Trạng thái: Sẵn sàng cho Axis 1, 2, 5 đọc.")
print(f"File xuất tại: {TRANSCRIPTS_PATH}")
